In [ ]:
"""
Matrix Transpose Triton Implementation (Day 7): Row-Major Transpose via Stride Multiplication

Math:
  Given A[m, n] for m ∈ [0, M), n ∈ [0, N), row-major layout, produce B[n, m]
  such that:
      B[n, m] = A[m, n]

Inputs / Outputs:
  A: [M, N] contiguous row-major tensor on CUDA device
  B: [N, M] contiguous row-major tensor on CUDA device

Assumptions:
  - A is 2D, contiguous, and on CUDA
  - M, N > 0
  - We expose explicit stride-based indexing, mirroring PyTorch-style logic:
        idx_A(m, n)   = m * stride_am + n * stride_an
        idx_B(n, m)   = n * stride_bm + m * stride_bn
    where for contiguous row-major tensors: stride_am = N, stride_an = 1, etc.

Parallel Strategy:
  - 2D grid of Triton programs over tiles of shape BLOCK_M × BLOCK_N
  - Each program transposes its tile using stride-based indexing

Mixed Precision Policy:
  - Same dtype for load/store; FP32 recommended for validation

Complexity:
  - FLOPs: ~0 (pure data movement)
  - Bytes moved: 2 * M * N * sizeof(dtype)

Test Vectors:
  - Deterministic random matrices for sizes like 32×32, 256×512, 513×1027
    with exact equality in FP32 (no arithmetic)
"""

import math
from typing import Tuple

import torch
import triton
import triton.language as tl


@triton.jit
def matrix_transpose_kernel(
    in_ptr,   # *const T, A[m, n]
    out_ptr,  # *mut T,   B[n, m]
    M,        # rows of A
    N,        # cols of A
    stride_am,
    stride_an,
    stride_bm,
    stride_bn,
    BLOCK_M: tl.constexpr,
    BLOCK_N: tl.constexpr,
):
    """
    2D tiled matrix transpose with explicit stride-based indexing.

    Layout (logical):
      - Input A has shape [M, N]
      - Output B has shape [N, M]

    Indexing:
      - A[m, n]   -> in_ptr  + m * stride_am + n * stride_an
      - B[n, m]   -> out_ptr + n * stride_bm + m * stride_bn
    """
    pid_m = tl.program_id(axis=0)  # tile index along rows (m)
    pid_n = tl.program_id(axis=1)  # tile index along cols (n)

    # Logical indices within the full matrices
    offs_m = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)[:, None]  # [BM, 1]
    offs_n = pid_n * BLOCK_N + tl.arange(0, BLOCK_N)[None, :]  # [1, BN]

    # Mask for in-bounds elements in A[M, N]
    mask = (offs_m < M) & (offs_n < N)

    # Input indices: A[m, n]
    in_indices = offs_m * stride_am + offs_n * stride_an
    a_tile = tl.load(in_ptr + in_indices, mask=mask, other=0.0)

    # Output indices: B[n, m] (note swapped logical indices)
    #   row index in B is n, col index is m
    outs_row = offs_n  # n
    outs_col = offs_m  # m
    out_indices = outs_row * stride_bm + outs_col * stride_bn

    tl.store(out_ptr + out_indices, a_tile, mask=mask)


def _validate_input(a: torch.Tensor) -> Tuple[int, int]:
    assert a.device.type == "cuda", "Input must be on CUDA device"
    assert a.dim() == 2, "Input must be 2D tensor [M, N]"
    assert a.is_contiguous(), "Input must be contiguous row-major"
    M, N = a.shape
    assert M > 0 and N > 0, "Matrix dimensions must be > 0"
    return M, N


def matrix_transpose_triton(
    a: torch.Tensor,
    *,
    block_m: int = 32,
    block_n: int = 32,
) -> torch.Tensor:
    """
    Public API: row-major matrix transpose via Triton.

    Args:
        a: Input tensor A with shape [M, N], contiguous, CUDA.
        block_m: Tile size along M dimension (rows).
        block_n: Tile size along N dimension (cols).

    Returns:
        Transposed tensor B with shape [N, M], contiguous row-major.
    """
    M, N = _validate_input(a)

    # Allocate output with transposed logical shape [N, M]
    b = torch.empty((N, M), device=a.device, dtype=a.dtype)

    grid = (
        triton.cdiv(M, block_m),  # number of tiles along M
        triton.cdiv(N, block_n),  # number of tiles along N
    )

    # Explicit strides (PyTorch-style):
    #   a_stride = (stride_am, stride_an)
    #   b_stride = (stride_bm, stride_bn)
    stride_am, stride_an = a.stride()
    stride_bm, stride_bn = b.stride()

    matrix_transpose_kernel[grid](
        a,
        b,
        M,
        N,
        stride_am,
        stride_an,
        stride_bm,
        stride_bn,
        BLOCK_M=block_m,
        BLOCK_N=block_n,
    )

    return b


def _run_correctness_tests() -> None:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if device.type != "cuda":
        print("CUDA not available; skipping Triton transpose tests.")
        return

    torch.manual_seed(7)

    dtypes = [torch.float32]
    shapes = [
        (32, 32),
        (64, 128),
        (256, 512),
        (513, 1027),  # edge case: not multiples of tile sizes
    ]

    for dtype in dtypes:
        for (M, N) in shapes:
            a = torch.randn(M, N, device=device, dtype=dtype)
            ref = a.t().contiguous()  # PyTorch transpose + materialize

            out = matrix_transpose_triton(a, block_m=32, block_n=32)
            torch.testing.assert_close(out, ref, rtol=0.0, atol=0.0)
            max_diff = (out - ref).abs().max().item()
            print(f"[Day7][OK] M={M}, N={N}, dtype={dtype}, max_diff={max_diff:.2e}")


def benchmark_matrix_transpose(
    shapes=None,
    dtype: torch.dtype = torch.float32,
) -> None:
    if shapes is None:
        shapes = [
            (256, 256),
            (512, 512),
            (1024, 1024),
            (2048, 1024),
        ]

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if device.type != "cuda":
        print("CUDA not available; skipping transpose benchmark.")
        return

    torch.manual_seed(123)

    for (M, N) in shapes:
        a = torch.randn(M, N, device=device, dtype=dtype)

        # Warmup
        for _ in range(10):
            _ = matrix_transpose_triton(a, block_m=32, block_n=32)
        torch.cuda.synchronize()

        iters = 100
        start_event = torch.cuda.Event(enable_timing=True)
        end_event = torch.cuda.Event(enable_timing=True)

        start_event.record()
        for _ in range(iters):
            out = matrix_transpose_triton(a, block_m=32, block_n=32)
        end_event.record()
        torch.cuda.synchronize()

        elapsed_ms = start_event.elapsed_time(end_event) / iters

        # Validate against PyTorch
        ref = a.t().contiguous()
        torch.testing.assert_close(out, ref, rtol=0.0, atol=0.0)

        numel = M * N
        bytes_moved = 2 * numel * a.element_size()  # read + write
        bandwidth_gb_s = (bytes_moved / (elapsed_ms / 1e3)) / 1e9

        print(
            f"[Day7][BENCH] M={M}, N={N}, dtype={dtype}, "
            f"time={elapsed_ms:.3f} ms, BW={bandwidth_gb_s:.2f} GB/s"
        )


if __name__ == "__main__":
    _run_correctness_tests()
    benchmark_matrix_transpose()